# Clustering: Customer Segmentation

In the first two notebooks, we worked on a supervised problem: predicting fraud, where every transaction in the training data was labelled as fraudulent or legitimate. The model learned from those labels.

Now the marketing team at Northgate Retail Bank has a different kind of question. They want to run targeted campaigns, but they have no predefined customer segments. There is no "correct grouping" column in the data. They need to **discover** structure rather than predict a known outcome.

This is **unsupervised learning**, and the explainer we just read introduced the key ideas: k-means clustering finds groups based on similarity, the algorithm *always* produces clusters (even from random noise), and it is our job to decide whether those clusters are meaningful. Let's put that into practice.

By the end of this notebook we will be able to:

- Prepare features for clustering with **StandardScaler**
- Apply **K-means** clustering to group customers
- Use the **elbow method** to choose the number of clusters
- Profile and interpret the resulting segments
- Visualise clusters using scatter plots and **PCA**

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_style("whitegrid")

---

## 1. Load and explore the data

We are working with a different dataset this time: customer profiles rather than individual transactions. Let's see what we have.

In [ ]:
df = pd.read_csv("../data/customer_profiles.csv")
print(f"{len(df)} customers")
df.head()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["age"], bins=25, edgecolor="white")
axes[0].set_xlabel("Age")
axes[0].set_title("Age distribution")

axes[1].hist(df["avg_monthly_spend"], bins=25, edgecolor="white")
axes[1].set_xlabel("Avg monthly spend (£)")
axes[1].set_title("Monthly spend distribution")

axes[2].hist(df["account_age_months"], bins=25, edgecolor="white")
axes[2].set_xlabel("Account age (months)")
axes[2].set_title("Account age distribution")

plt.tight_layout()
plt.show()

The distributions hint that there may be distinct groups hiding in this data. Customers are not uniformly spread — there appear to be clusters of younger lower-spenders and older higher-spenders. Let's see if k-means can tease those groups apart.

---

## 2. Feature selection and scaling

K-means works by measuring the distance between data points. This creates a practical problem: if one feature ranges from 1 to 240 (account age in months) and another ranges from 1 to 6 (number of products), the large-scale feature will dominate the distance calculation and the smaller feature will barely register.

**StandardScaler** fixes this by transforming each feature to have mean 0 and standard deviation 1, putting all features on an equal footing.

In [ ]:
# Select numeric features for clustering
cluster_features = ["age", "account_age_months", "num_products", "avg_monthly_spend"]
X = df[cluster_features].copy()

print("Before scaling:")
print(X.describe().round(1))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=cluster_features)
print("After scaling:")
print(X_scaled_df.describe().round(2))

Each feature now has mean ~0 and standard deviation ~1. The values represent how many standard deviations each observation sits from the mean, giving the algorithm a common scale for all features.

---

## 3. The elbow method: choosing k

The explainer noted that the analyst must choose **k** (the number of clusters) before running k-means. This is a genuine decision, not a formality. We try a range of values and look at the **inertia** (total within-cluster distance). As k increases, inertia always decreases — more clusters means each point is closer to its nearest centre. We look for an "elbow" where adding more clusters stops producing a meaningful reduction in inertia.

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_range, inertias, "o-")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow method")
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

The curve should show a bend around k=4. The improvement from k=4 to k=5 is much smaller than from k=3 to k=4, which suggests **4 clusters** is a reasonable starting point. Remember, though, that this is a guideline rather than a proof. We will need to check whether four clusters actually make sense when we profile them.

---

## 4. Fit K-means with chosen k

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(df["cluster"].value_counts().sort_index())

Each customer now belongs to a cluster (0, 1, 2, or 3). The labels are arbitrary — "cluster 0" has no inherent meaning. The algorithm found four groups; our job is to work out what they represent.

---

## 5. Examine cluster centres

The **cluster centres** (centroids) are the mean values of each feature for each cluster. Since we scaled the features before clustering, we need to inverse-transform the centres to see them in the original units.

In [ ]:
centres_scaled = kmeans.cluster_centers_
centres = scaler.inverse_transform(centres_scaled)

centres_df = pd.DataFrame(centres, columns=cluster_features)
centres_df.index.name = "Cluster"
centres_df.round(1)

Each row describes the "average customer" in that cluster. Look for distinguishing characteristics: one cluster might have younger customers with lower spend; another might be older with high product counts. These differences are what will make the segments useful to the marketing team.

---

## 6. Profile the clusters

The cluster centres give us a starting point, but a richer picture comes from computing grouped statistics across the full dataframe — including categorical columns like income bracket and region that were not used in clustering itself.

In [ ]:
profile = df.groupby("cluster").agg(
    count=("customer_id", "count"),
    avg_age=("age", "mean"),
    avg_account_months=("account_age_months", "mean"),
    avg_products=("num_products", "mean"),
    avg_spend=("avg_monthly_spend", "mean"),
).round(1)

profile

In [ ]:
# Income distribution per cluster
income_dist = pd.crosstab(df["cluster"], df["income_bracket"], normalize="index").round(2)
income_dist

In [ ]:
# Region distribution per cluster
region_dist = pd.crosstab(df["cluster"], df["region"], normalize="index").round(2)
region_dist

If regions are evenly distributed across clusters, location is not a defining characteristic of these segments. That is useful information too — it tells the marketing team that these groups are distinguished by behaviour, not geography.

This is where the domain judgement the explainer described comes in. Can we describe each cluster in a sentence the marketing team would find useful? A cluster of younger customers with low spend and few products might be "Young Professionals." A cluster of older customers with long tenure and high product counts might be "Long-standing Retirees." If we can name them, they are probably worth acting on.

---

## 7. Visualise clusters: scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = sns.color_palette("tab10", 4)

for c in range(4):
    mask = df["cluster"] == c
    axes[0].scatter(df.loc[mask, "age"], df.loc[mask, "avg_monthly_spend"],
                    alpha=0.5, label=f"Cluster {c}", color=palette[c], s=20)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Avg monthly spend (£)")
axes[0].set_title("Age vs Monthly Spend")
axes[0].legend()

for c in range(4):
    mask = df["cluster"] == c
    axes[1].scatter(df.loc[mask, "account_age_months"], df.loc[mask, "num_products"],
                    alpha=0.5, label=f"Cluster {c}", color=palette[c], s=20)
axes[1].set_xlabel("Account age (months)")
axes[1].set_ylabel("Number of products")
axes[1].set_title("Account Age vs Products")
axes[1].legend()

plt.tight_layout()
plt.show()

Scatter plots using two features at a time give a partial view. Our clusters were defined by four features simultaneously, so we may not see clean separation in any single two-dimensional slice. That does not mean the clusters are bad; it means we need a way to view all four dimensions at once. That is where PCA comes in.

---

## 8. PCA for 2D projection

**Principal Component Analysis (PCA)** reduces the four scaled features down to two **principal components** that capture as much of the overall variance as possible. This gives us a single 2D view that accounts for information from all features, rather than just two at a time.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Total variance explained:  {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for c in range(4):
    mask = df["cluster"] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               alpha=0.5, label=f"Cluster {c}", color=palette[c], s=20)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)")
ax.set_title("Customer clusters (PCA projection)")
ax.legend()
plt.tight_layout()
plt.show()

The PCA projection shows how well-separated the clusters are when all four features are combined into two dimensions. Some overlap is normal and expected in real customer data — people do not sort neatly into boxes.

---

## 9. Interpreting the results

The explainer made an important point: clustering always produces groups, so the question is never "did the algorithm find clusters?" but "are these clusters worth acting on?" Here are the questions to ask:

- Do the clusters correspond to recognisable customer types that the marketing team would name?
- Would the team design genuinely different campaigns for each segment?
- Are the clusters stable if we change k or the features?

There is no precision or recall score to tell us whether the answer is good. Clustering is an exploratory tool, and its value comes from the conversations it starts with stakeholders, not from a number on a test set.

---

## 10. Exercises

### Exercise 1: Try k=3 and k=5

Refit K-means with k=3 and k=5. For each, compute the cluster profiles (mean age, mean spend, mean products) using `groupby`. How do the segments change? Does k=3 merge two groups that k=4 kept separate? Does k=5 split a group into two similar halves?

In [ ]:
# Your code here


### Exercise 2: Include income as a feature

Convert `income_bracket` into a numeric value (e.g., low=1, medium=2, high=3, very_high=4). Add it to the feature set, re-scale, and refit K-means with k=4. Compare the cluster profiles to the original. Does adding income change the segments?

In [ ]:
# Your code here


### Exercise 3: Silhouette score

Import `silhouette_score` from `sklearn.metrics`. Compute it for k=2 through k=8. The silhouette score ranges from -1 to 1; higher is better (points are well-matched to their own cluster and poorly matched to neighbours). Plot k vs silhouette score. Which k scores highest?

In [ ]:
# Your code here


### Exercise 4: Name the segments

Using the k=4 cluster profiles from section 6, assign a descriptive marketing name to each cluster (e.g., "Budget-Conscious Newcomers", "Established Families"). Create a new column `segment_name` in the dataframe that maps cluster numbers to your chosen names. Print the first few rows showing `customer_id`, `cluster`, and `segment_name`.

In [ ]:
# Your code here


---

## Summary

In this notebook we moved from supervised classification to unsupervised clustering, tackling a problem where no labels exist. We:

- Prepared features for clustering by applying **StandardScaler** to put all features on the same scale
- Used the **elbow method** (inertia plot) to choose an appropriate number of clusters
- Fitted **K-means** clustering and assigned each customer to a segment
- Examined **cluster centres** and profiled segments with `groupby` statistics
- Visualised clusters with **scatter plots** on pairs of features
- Applied **PCA** to project all features into two dimensions for a combined view
- Discussed how to judge whether clusters are meaningful and actionable

The key difference from the previous two notebooks is the absence of a right answer. In classification, the confusion matrix tells us exactly where the model succeeds and fails. In clustering, that judgement falls to us and the domain experts we work with. The algorithm finds structure; we decide whether it means anything.